In [1]:
# Cell 1 — Load cleaned support pairs

import pandas as pd

support_pairs_df = pd.read_csv(
    "../data/processed/support_pairs.csv"
)

print("Support pairs loaded:", len(support_pairs_df))
print("\nColumns:")
print(support_pairs_df.columns.tolist())

print("\nSample:")
display(support_pairs_df.head())

Support pairs loaded: 3092

Columns:
['conversation_id', 'customer_tweet_id', 'support_tweet_id', 'customer_message', 'support_response', 'response_behavior', 'response_quality']

Sample:


,conversation_id,customer_tweet_id,support_tweet_id,customer_message,support_response,response_behavior,response_quality
0,100245,100245,100244,What is this ? since I upgraded with this $hi&...,@137967 We'd like to help. Does this seem to p...,troubleshooting,guided_escalation
1,100245,100243,100241,I have tried restarting my macbook but it keep...,"@137967 Either way, let us know what happens v...",dm_escalation,guided_escalation
2,100540,100540,100539,My iPhone been moving slow af the past couple ...,@137986 We’d love to help with the performance...,troubleshooting,guided_escalation
3,101228,101228,101227,"Dear god not again,",@138151 We want to make sure your words and te...,troubleshooting,guided_escalation
4,101231,101231,101230,Noticed a bug on my @115858 iPhone X/iOS 11.1....,@138152 We can assist you. Thanks for bringing...,troubleshooting,actionable


In [2]:
# Cell 2 — Prepare customer messages for TF-IDF

customer_messages = (
    support_pairs_df["customer_message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove completely empty messages
customer_messages = customer_messages[
    customer_messages.str.len() > 0
].reset_index(drop=True)

print("Customer messages:", len(customer_messages))
print("Unique messages:", customer_messages.str.lower().nunique())

print("\nSample customer messages:")
for i, message in enumerate(customer_messages.head(10), start=1):
    print(f"{i}. {message}")

Customer messages: 3092
Unique messages: 3023

Sample customer messages:
1. What is this ? since I upgraded with this $hi&*y macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!
2. I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.
3. My iPhone been moving slow af the past couple weeks. I need answers
4. Dear god not again,
5. Noticed a bug on my @115858 iPhone X/iOS 11.1.2 While I’m on the phone I can’t close apps.
6. leg dit even uit? Dit is mijn oud e-mail adress/apple id, sinds een recente update krijg ik constant deze melding. Ik heb sinds paar jaar een nieuw e-mail adres en ook een apple id op dit adres. Ik was (nog altijd) het wachtwoord kwijt van het oude e-mail & apple-id
7. Since a recent update i’m constantly getting these notifications about my old apple ID /e-mail adress, wich i’ve 

In [3]:
# Cell 3 — Build TF-IDF representation

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(customer_messages)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of features:", len(vectorizer.get_feature_names_out()))

print("\nFirst 20 features:")
print(vectorizer.get_feature_names_out()[:20])

TF-IDF matrix shape: (3092, 2143)
Number of features: 2143

First 20 features:
['000' '01' '03' '10' '10 10' '10 11' '10 12' '10 13' '10 seconds'
 '10 times' '100' '11' '11 03' '11 11' '11 115858' '11 15b202' '11 apps'
 '11 battery' '11 beta' '11 bug']


In [4]:
# Cell 4 — Inspect important TF-IDF terms

import numpy as np

feature_names = vectorizer.get_feature_names_out()

# Calculate average TF-IDF score of each feature across all messages
mean_tfidf = np.asarray(
    tfidf_matrix.mean(axis=0)
).ravel()

top_indices = mean_tfidf.argsort()[::-1][:50]

top_terms = pd.DataFrame({
    "term": feature_names[top_indices],
    "mean_tfidf": mean_tfidf[top_indices]
})

print("Top 50 TF-IDF terms:")
display(top_terms)

Top 50 TF-IDF terms:


,term,mean_tfidf
0,11,0.044040
1,115858,0.029847
2,iphone,0.029675
3,phone,0.027280
4,ios,0.026149
5,update,0.022285
6,ios 11,0.020063
7,fix,0.018260
8,help,0.017758
9,just,0.016462


In [5]:
# Cell 5 — Discover latent topics from TF-IDF

from sklearn.decomposition import NMF

# Start with more topics than our final taxonomy.
# This helps us discover smaller issues before combining them.
n_topics = 12

nmf_model = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

nmf_matrix = nmf_model.fit_transform(tfidf_matrix)

print("NMF matrix shape:", nmf_matrix.shape)
print("Number of discovered topics:", n_topics)

NMF matrix shape: (3092, 12)
Number of discovered topics: 12


In [6]:
# Cell 6 — Inspect top terms for each discovered topic

feature_names = vectorizer.get_feature_names_out()

n_top_words = 12

for topic_idx, topic in enumerate(nmf_model.components_, start=1):
    top_indices = topic.argsort()[::-1][:n_top_words]
    top_terms = [feature_names[i] for i in top_indices]

    print(f"Topic {topic_idx}:")
    print("  " + " | ".join(top_terms))
    print()
    

Topic 1:
  11 | version | version 11 | ios 11 | iphone 11 | 11 11 | 6s | updated 11 | started | updated | 6s 11 | says

Topic 2:
  fix | 115858 | 115858 fix | https | shit | letter | type | problem | wtf | question | going | fix shit

Topic 3:
  ios | ios 11 | iphone ios | battery | 10 | ios 10 | updated ios | plus ios | using | using ios | new ios | bug

Topic 4:
  iphone | plus | iphone plus | 6s | iphone 6s | battery | hi | does | iphone ios | bluetooth | new iphone | iphone 11

Topic 5:
  help | need | need help | didn | hi | didn help | mac | tried | guys | password | keyboard | won

Topic 6:
  phone | updated | just | updated phone | 115858 | like | doing | just updated | happening | freezing | shit | ve

Topic 7:
  yes | did | doing | just | yes just | yes able | able | ipad | times | service | sms | yes update

Topic 8:
  thank | know | worked | problem | fixed thank | notes | response | contact | omg | fixed | reply | support

Topic 9:
  applesupport | https | applesupport htt

In [7]:
# Cell 7 — Clean noisy tokens for intent discovery

import re

def clean_for_intent_discovery(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove Twitter-style mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove long numeric IDs
    text = re.sub(r"\b\d{5,}\b", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

customer_messages_clean = customer_messages.apply(
    clean_for_intent_discovery
)

print("Original messages:", len(customer_messages))
print("Cleaned messages:", len(customer_messages_clean))

print("\nBefore:")
print(customer_messages.iloc[0])

print("\nAfter:")
print(customer_messages_clean.iloc[0])

Original messages: 3092
Cleaned messages: 3092

Before:
What is this ? since I upgraded with this $hi&*y macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!

After:
what is this ? since i upgraded with this $hi&*y macos my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying apple macbook now!


In [8]:
# Cell 8 — Build cleaner TF-IDF representation

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(customer_messages_clean)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of features:", len(vectorizer.get_feature_names_out()))

print("\nFirst 20 features:")
print(vectorizer.get_feature_names_out()[:20])

TF-IDF matrix shape: (3092, 2089)
Number of features: 2089

First 20 features:
['000' '01' '03' '10' '10 10' '10 11' '10 12' '10 13' '10 seconds'
 '10 times' '100' '11' '11 03' '11 11' '11 15b202' '11 apps' '11 battery'
 '11 beta' '11 bug' '11 fix']


In [9]:
# Cell 9 — Discover topics from cleaned TF-IDF

from sklearn.decomposition import NMF

n_topics = 12

nmf_model = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

nmf_matrix = nmf_model.fit_transform(tfidf_matrix)

print("NMF matrix shape:", nmf_matrix.shape)
print("Number of discovered topics:", n_topics)

NMF matrix shape: (3092, 12)
Number of discovered topics: 12


In [10]:
# Cell 10 — Inspect top terms for each cleaned topic

feature_names = vectorizer.get_feature_names_out()
n_top_words = 12

for topic_idx, topic in enumerate(nmf_model.components_, start=1):
    top_indices = topic.argsort()[::-1][:n_top_words]
    top_terms = [feature_names[i] for i in top_indices]

    print(f"Topic {topic_idx}:")
    print("  " + " | ".join(top_terms))
    print()

Topic 1:
  11 | version | version 11 | ios 11 | iphone 11 | 11 11 | 6s | updated 11 | started | updated | 6s 11 | says

Topic 2:
  phone | updated | just | updated phone | shit | like | fuck | freezing | just updated | happening | doing | charge

Topic 3:
  ios | ios 11 | iphone ios | battery | 10 | ios 10 | updated ios | plus ios | using | new ios | bug | using ios

Topic 4:
  help | need | need help | didn | didn help | hi | mac | tried | guys | password | pls help | won

Topic 5:
  fix | fix shit | shit | bug | problem | glitch | need fix | fix phone | need | issue | fix problem | annoying

Topic 6:
  iphone | plus | iphone plus | 6s | iphone 6s | battery | does | hi | iphone ios | bluetooth | new iphone | does iphone

Topic 7:
  yes | did | doing | just | yes just | yes able | able | ipad | times | yes update | service | sms

Topic 8:
  thank | know | worked | problem | fixed thank | notes | response | contact | omg | fixed | reply | slower

Topic 9:
  apple | music | app | apple m

In [11]:
# Cell 11 — Inspect representative customer messages for each NMF topic

n_examples = 5

for topic_idx in range(n_topics):
    print("=" * 80)
    print(f"TOPIC {topic_idx + 1}")
    print("=" * 80)

    # Get messages with the highest NMF score for this topic
    top_message_indices = np.argsort(
        nmf_matrix[:, topic_idx]
    )[::-1][:n_examples]

    for rank, idx in enumerate(top_message_indices, start=1):
        print(f"\n{rank}. {customer_messages_clean.iloc[idx]}")
    
    print()

TOPIC 1

1. 11.2

2. 11.1.2

3. 11.1.2

4. 11.1.1

5. 11.1.1

TOPIC 2

1. “phone”

2. after i updated my phone.

3. since i updated my phone my phone has been tripping

4. why cant i️ tupe the letter i️ i️ updated my phone.

5. i just updated my phone again and still can’t text i

TOPIC 3

1. ios 11.1

2. ios 11

3. ios 11.0.3

4. ios 11.1

5. it’s ios 11.1.1

TOPIC 4

1. please help us lmao

2. 🗣help usssss

3. can you please help me

4. please help me???

5. sameeee wtfff please help us

TOPIC 5

1. fix this i️ !!!!

2. now fix this “i.t” 🙄

3. can you fix this 👉🏼i️m

4. fix this now!!!!! i️ i️ i️ i️ i️ i️ i️ i️ i️ i️ i️

5. fix it

TOPIC 6

1. iphone 6

2. iphone 7

3. iphone 7+

4. iphone 7

5. can you fix my iphone please.

TOPIC 7

1. yes

2. yes

3. yes.

4. yes...

5. yes i'm.

TOPIC 8

1. thank you very much!

2. thank you!

3. thank you

4. thank you!!

5. okey. thank you!

TOPIC 9

1. why music app is forcing me to subscribe apple music

2. apple music.

3. no app updates. a

In [12]:
# Cell 12 — Save TF-IDF and NMF artifacts

import os
import joblib

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save TF-IDF matrix
from scipy.sparse import save_npz

save_npz(
    "../data/processed/customer_tfidf.npz",
    tfidf_matrix
)

# Save vectorizer
joblib.dump(
    vectorizer,
    "../models/tfidf_vectorizer.joblib"
)

# Save NMF model
joblib.dump(
    nmf_model,
    "../models/nmf_intent_discovery.joblib"
)

# Save NMF topic representation
np.save(
    "../data/processed/nmf_topic_matrix.npy",
    nmf_matrix
)

print("Intent discovery artifacts saved successfully.")
print("TF-IDF matrix:", tfidf_matrix.shape)
print("NMF matrix:", nmf_matrix.shape)

Intent discovery artifacts saved successfully.
TF-IDF matrix: (3092, 2089)
NMF matrix: (3092, 12)
